# Generate TEM-MCQ

This notebook generates the TEM-MCQ evaluation dataset using GPT-4.1.

For each TEM subfigure in the test split, GPT receives the image and its parent figure caption, and generates 9 multiple-choice questions across three categories and three difficulty levels:

| Category | Focus |
|----------|-------|
| **Visual Perception** | Shape, texture, contrast, size, distribution |
| **Scientific Reasoning** | Crystal structure, defect identification, phase relationships |
| **Experimental Methodology** | Imaging mode, artifacts, sample preparation |

# Import packages

In [1]:
import os
import gc
import pandas as pd
import base64
import csv
from pathlib import Path
from openai import OpenAI

## Settings

Adjust the paths below to match your local setup:

- `FIGURE_DIR`: directory containing TEM subfigures
- `INPUT_CSV_PATH`: test split CSV containing `CROP_IMAGE` and `CAPTION` columns (output of `dataset_split.ipynb`)
- `OUTPUT_CSV_PATH`: output CSV where generated MCQs will be saved

The OpenAI API key is read from the `OPENAI_API_KEY` environment variable.

In [ ]:
FIGURE_DIR      = "/path/to/your/TEM_figures"
INPUT_CSV_PATH  = "/path/to/your/test.csv"
OUTPUT_CSV_PATH = "/path/to/your/TEM_MCQ.csv"

API_Key = os.getenv("OPENAI_API_KEY")
client  = OpenAI(api_key=API_Key)

## Define Prompts

In [3]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

In [ ]:
def test_question_system_prompt():
    return """
You are a senior scientific assessment designer for vision-language models (VLMs) in TEM microscopy.
Your task is to generate evaluation questions that TEST a model's ability to interpret TEM images based on visual evidence.

### CRITICAL EVALUATION CONTEXT: "THE BLIND TEST"
The model taking this test will **NOT** see the original caption. It sees **ONLY the IMAGE and your QUESTION**.
Therefore, you must bridge the "knowledge gap" by injecting necessary context into the question stem.

### GLOBAL RULES:
1. **Visual Solvability:** The ANSWER must be derived strictly from visual features (morphology, contrast, lattice, artifacts) visible in the image.
2. **Context Injection:**
   - If the answer requires knowing the material name (e.g., "What defect is in this GaN crystal?"), you MUST state "In this GaN crystal..." in the question.
   - Do NOT ask "What material is this?" unless the material has a unique, unmistakable visual fingerprint.
3. **No Invisible Properties:** Even with context, do not ask about properties invisible in the image (e.g., conductivity, melting point, exact doping %).
4. **Structure:**
   - All questions must be Multiple Choice (4 options: A, B, C, D).
   - 1 Correct Answer, 3 Plausible Distractors.
5. **Taxonomy:** Follow the framework:
   - **Category 1: Visual Perception** (Direct observation)
   - **Category 2: Scientific Reasoning** (Inference based on visual evidence + injected context)
   - **Category 3: Experimental Methodology** (Imaging technique, artifacts, quality)
6. **Difficulty:** Generate 1 Easy, 1 Medium, 1 Hard question per category (Total 9).

### DISTRACTOR GUIDELINES:
- **Plausible:** Wrong options must be scientifically sound terms but visually incorrect for *this* specific image.
- **No Hallucinations:** Do not create "trick" questions that rely on pixel-peeping non-existent features.
"""

In [ ]:
def build_test_question_prompt(caption):
    return f"""
You will receive:
1. **Input Image:** A specific CROP from a parent figure.
2. **Input Caption:** The text describing the WHOLE parent figure.

**Goal:** Generate 9 MCQs (3 Categories x 3 Difficulties) for a model that DOES NOT see the caption.

----------------------
### CATEGORY DEFINITIONS:

**1. Visual Perception (Observation)**
   - *Focus:* Shape, texture, contrast, size, distribution.
   - *Context Strategy:* Usually requires minimal context.
   - *Example:* "What is the predominant morphology of the dark particles?"

**2. Scientific Reasoning (Inference)**
   - *Focus:* Crystal structure, defect identification, growth mechanisms, phase relationships.
   - *Context Strategy:* **HIGH.** You likely need to state the material name to allow the model to reason about its properties.
   - *Example:* "Given that this is a **Zeolite** sample, what does the pore structure suggest about its orientation?"

**3. Experimental Methodology (Methodology)**
   - *Focus:* Imaging mode (BF/DF/STEM), beam damage, sample preparation artifacts, focus/astigmatism.
   - *Example:* "The bright halo around the particles suggests which imaging artifact?"

----------------------
### GENERATION PROTOCOL:

1. **Analyze Caption:** Extract facts (Material System, Synthesis Method, Imaging Mode).
2. **Visual Check:** Verify which facts are actually visible in the image.
3. **Context Injection Decision:**
   - Ask yourself: "If I remove the caption, can I answer this?"
   - If NO -> Add the material name or condition into the Question Stem.
4. **Draft Options:** Ensure distractors are visually distinct (e.g., if the answer is "Spherical", distractors should be "Cubic", "Rod-like", "Plate-like").

----------------------
### OUTPUT SCHEMA (Strict JSON):

{{
  "reasoning_process": {{
    "visual_audit": "List 3 visible features...",
    "caption_facts_to_inject": "Material name, specific orientation...",
    "imaging_mode_detected": "HRTEM / STEM / SAED..."
  }},
  "visual_perception": [
    {{
      "difficulty": "easy",
      "question": "The question text (with context if needed)...",
      "options": {{
        "A": "Option text",
        "B": "Option text",
        "C": "Option text",
        "D": "Option text"
      }},
      "correct_answer": "A",
      "rationale": "Explain why visual evidence supports A and rules out others."
    }}
    // ... 2 more (medium, hard)
  ],
  "reasoning": [
    // ... 3 questions
  ],
  "experimental_design": [
    // ... 3 questions
  ]
}}

----------------------
Caption:
\"\"\"
{caption}
\"\"\"
"""

## Check Progress

Loads already-processed images from `OUTPUT_CSV_PATH` so the notebook can be safely resumed after interruption.

In [ ]:
done_images = set()
if os.path.exists(OUTPUT_CSV_PATH):
    with open(OUTPUT_CSV_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        for row in reader:
            if row:
                done_images.add(row[0])
print(f"{len(done_images)} images done")

## Generate MCQs

For each subfigure in `INPUT_CSV_PATH`:
1. Skips images already processed
2. Encodes the image in base64 and sends it to GPT-4.1 with the parent caption
3. Saves the generated MCQs incrementally to `OUTPUT_CSV_PATH`

In [ ]:
df = pd.read_csv(INPUT_CSV_PATH, dtype=str, encoding="utf-8")

file_exists = os.path.exists(OUTPUT_CSV_PATH)

with open(OUTPUT_CSV_PATH, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(["CROP_IMAGE", "RESPONSE"])
    
    for idx, row in df.iterrows():
        image_name = row['CROP_IMAGE']

        if image_name in done_images:
            continue
    
        print(f"Dealing with {idx}th image: {row['CROP_IMAGE']} ")
        
        image_path = FIGURE_DIR / image_name
        caption = row['CAPTION']
        base64_image = encode_image(image_path)

        try:
            response = client.responses.create(
                model="gpt-4.1",
                temperature=0.25,
                input=[
                    {
                        "role": "system",
                        "content": [{"type": "input_text", "text": test_question_system_prompt()}],
                    },
                    {
                        "role": "user",
                        "content": [
                            {"type": "input_text", "text": build_test_question_prompt(caption)},
                            {
                                "type": "input_image",
                                "image_url": f"data:image/png;base64,{base64_image}",
                            },
                        ],
                    },
                ],
            )

            output = response.output_text
            writer.writerow([row['CROP_IMAGE'],output])
            f.flush()
            os.fsync(f.fileno())

            done_images.add(image_name)
            print(f"Save: {image_name}")
        except Exception as e:
            print(f"Error on {image_name}: {e}")
            continue
            
        del base64_image
        del response
        gc.collect()